In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt

# Exact solution versus untruncated solution

The mibitrans solution uses the exact solution of the 3D transport ADE, as described in Wexler et al. (1992). The anatrans solution approximates this solution by the method described in Domenico (1987), but without truncating the x part of the equation. Both are adapted for multiple source zones and source depletion, see equations below.

## Anatrans solution
\begin{align}\tag{1}
    C(x, y, z, t) &= \sum_{i=1}^{n}\left\{ \frac{C^*_{0,i}}{8} \exp \left(-\gamma_s t\right) \right. \\
    &\quad \cdot \left( \exp \left[ \frac{x\left(1-P\right)}{2\alpha_x}\right]
    \cdot \operatorname{erfc} \left[ \frac{x - Pvt}{2\sqrt{\alpha_x vt }} \right] \right.\\
    &\quad \:  + \left.  \exp \left[ \frac{x\left(1+P\right)}{2\alpha_x}\right]
    \cdot \operatorname{erfc} \left[ \frac{x + Pvt}{2\sqrt{\alpha_x vt }} \right] \right) \\
    &\quad \cdot \left( \operatorname{erf} \left[ \frac{y + Y_i/2}{2\sqrt{\alpha_y x}} \right] - \operatorname{erf} \left[ \frac{y - Y_i/2}{2\sqrt{\alpha_y x)}} \right] \right) \\
    &\quad \cdot \left. \left( \operatorname{erf} \left[ \frac{Z}{2\sqrt{\alpha_z x)}} \right] - \operatorname{erf} \left[ \frac{-Z}{2\sqrt{\alpha_z x}} \right] \right) \right\} \\
    & \text{with} \quad P = \sqrt{1+4\left(\mu - \gamma_s \right) \alpha_x/v}
\end{align}


## Mibitrans solution
$$
\begin{equation}\tag{2}
\begin{aligned}
    C(x,y,z,t) &= \sum_{i=1}^{n}\left(C^*_{0,i}\frac{x}{8\sqrt{\pi \alpha_{x}\frac{v\tau}{R}}}\exp(-\gamma_s t) \right. \\
    &\quad \cdot \int_{0}^{t}\left[\frac{1}{\tau^{\frac{3}{2}}} \exp\left((\gamma_s - \mu)\tau - \frac{(x-\frac{v\tau}{R})^2}{4\alpha_{x}\frac{v\tau}{R}}\right) \right. \\
    &\quad \cdot \left\{\operatorname{erfc}\left(\frac{y-Y_i}{2 \sqrt{\alpha_{y}\frac{v\tau}{R}}}\right)-\operatorname{erfc}\left(\frac{y+Y_i}{2 \sqrt{\alpha_{y}\frac{v\tau}{R}}}\right) \right\} \\
    &\quad \left. \left. \cdot \left\{\operatorname{erfc}\left(\frac{-Z}{2 \sqrt{\alpha_{z}\frac{v\tau}{R}}}\right)-\operatorname{erfc}\left(\frac{Z}{2 \sqrt{\alpha_{z}\frac{v\tau}{R}}}\right) \right\}\right] d\tau \right)
\end{aligned}
\end{equation}
$$

The approximation (see notebook `example_validity_substitution`, for more information) for the transverse dispersivity terms in the Anatrans solution introduces an error, of which the size depends on parameter choices. For the Domenico (1987) (Bioscreen in this package) solution specifically, the error size is dependent on the Peclet number defined as $Pe = \frac{vx_R}{D_x}$, where $x_R$ is the characteristic travel distance (Guyonnet and Neville, 2004). $D_x$ is the longitudinal dispersion coefficient, defined as $D_{eff} + \alpha_x v$. If diffusion is considered to be negligible, the expression becomes, $Pe = \frac{x_R}{\alpha_x}$, and thus is mainly dependent on the dispersivity. For the Anatrans solution however, the longitudinal component of the equation is nearly identical to that of the Mibitrans solution. Thus, if $\alpha_y$ and $\alpha_z = 0$, the Mibitrans and Anatrans solutions resolve to the same concentration distribution. Which is shown below.

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity=0.277, # Flow velocity [m/d]
    porosity=0.25,  # Effective soil porosity [-]
    alpha_x=10,     # Longitudinal dispersivity, in [m]
    alpha_y=0,      # Transverse horizontal dispersivity, in [m]
    alpha_z=0,    # Transverse vertical dispersivity, in [m]
    diffusion=0, # Molecular diffusion, in [m2/day]
)

att = mbt.AttenuationParameters(
    retardation=1,
    # Contaminant half life, in [days]
    half_life=0
)

source = mbt.SourceParameters(
    source_zone_boundary=np.array([10]),
    source_zone_concentration=np.array([11]),
    depth=2.5,
    total_mass="inf"
)

model = mbt.ModelParameters(
    # Model extent in the longitudinal (x) direction in [m].
    model_length = 800,
    # Model extent in the transverse horizontal (y) direction in [m].
    model_width = 150,
    # Model duration in [days].
    model_time = 10*365,
    # Model grid discretization step size in the longitudinal (x) direction, in [m].
    dx = 2,
    # Model grid discretization step size in the transverse horizontal (y) direction, in [m].
    dy = 1,
    # Model time discretization step size, in [days]
    dt = 365 / 5
)

mbt_object = mbt.Mibitrans(hydro, att, source, model)
mbt_results_longitudinal = mbt_object.run()

ana_object = mbt.Anatrans(hydro, att, source, model)
ana_results_longitudinal = ana_object.run()

bio_object = mbt.Bioscreen(hydro, att, source, model)
bio_results_longitudinal = bio_object.run()

In [ ]:
times=np.array([365, 2*365, 4*365, 6*365])
colors_ana = ["darkblue", "blue", "royalblue", "cornflowerblue", "skyblue"]
colors_mbt = ["darkgreen", "forestgreen", "limegreen", "greenyellow", "palegreen"]
colors_bio = ["maroon", "red", "orangered", "tomato", "coral"]

for i in range(len(times)):
    mbt_results_longitudinal.centerline(
        time=times[i],
        color=colors_mbt[i],
        lw=3,
        alpha=0.7,
        label=f"mbt, $t={times[i]}d$"
    )
    ana_results_longitudinal.centerline(
        time=times[i],
        color=colors_ana[i],
        lw=2,
        linestyle="--",
        label=f"ana, $t={times[i]}$d"
    )
    bio_results_longitudinal.centerline(
        time=times[i],
        color=colors_bio[i],
        lw=2,
        linestyle=":",
        label=f"bio, $t={times[i]}$d"
    )
    plt.legend(bbox_to_anchor=(1,1))

plt.show()

As shown, the Bioscreen solution has a slight error compared to the Anatrans and Mibitrans solutions, caused by the truncation in the Domenico (1987) solution.. Thus, difference between Anatrans and Mibitrans is caused by the transverse terms.

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity=0.277, # Flow velocity [m/d]
    porosity=0.25,  # Effective soil porosity [-]
    alpha_x=10,     # Longitudinal dispersivity, in [m]
    alpha_y=0.5,      # Transverse horizontal dispersivity, in [m]
    alpha_z=0.001,    # Transverse vertical dispersivity, in [m]
    diffusion=0, # Molecular diffusion, in [m2/day]
)

mbt_object = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_object.run()

ana_object = mbt.Anatrans(hydro, att, source, model)
ana_results = ana_object.run()

In [ ]:
times=np.array([365, 2*365, 4*365, 6*365])

for i in range(len(times)):
    mbt_results.centerline(
        time=times[i],
        color=colors_mbt[i],
        lw=3,
        label=f"mbt, $t={times[i]}d$"
    )
    ana_results.centerline(
        time=times[i],
        color=colors_ana[i],
        lw=2,
        alpha=0.8,
        linestyle="--",
        label=f"ana, $t={times[i]}$d"
    )

plt.legend()
plt.show()

In [ ]:
times=np.array([1*365, 2*365, 3*365, 4*365, 5*365])

for i in range(len(times)):
    mbt_results.transverse(
        x_position = 300,
        time=times[i],
        color=colors_mbt[i],
        lw=3,
        label=f"mbt, $t={times[i]}d$"
    )
    ana_results.transverse(
        x_position = 300,
        time=times[i],
        color=colors_ana[i],
        lw=2,
        linestyle="--",
        label=f"ana, $t={times[i]}d$"
    )

plt.ylim((0, 5))
plt.xlim((-50,50))
plt.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
x_positions = [75, 150, 300, 600]

for i in range(len(x_positions)):
    mbt_results.breakthrough(
        x_position=x_positions[i],
        color=colors_mbt[i],
        lw=3,
        label=f"mbt, $x={x_positions[i]}m$"
    )
    ana_results.breakthrough(
        x_position=x_positions[i],
        color=colors_ana[i],
        lw=2,
        linestyle="--",
        label=f"ana, $x={x_positions[i]}m$"
    )

plt.title("Breakthrough curve of Mibitrans and Anatrans model, at various x locations.")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

Under these transport parameters, the Anatrans model consistently underestimates the contaminant concentrations at the plume centerline. The transverse distribution shows that Anatrans as increased concentrations compared to Mibitrans at the fringes of the plume. Thus, the approximation used in the Anatrans solution overestimates the effect of transverse dispersion on the concentration distribution.

#### Comparing model differences for various flow velocities and dispersivities

Assuming that diffusion is negligible, the flow velocity is of no influence on the differences between the Anatrans and Mibitrans models.

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity=0.1, # Flow velocity [m/d]
    porosity=0.25,  # Effective soil porosity [-]
    alpha_x=10,     # Longitudinal dispersivity, in [m]
    alpha_y=0.5,      # Transverse horizontal dispersivity, in [m]
    alpha_z=0.001,    # Transverse vertical dispersivity, in [m]
    diffusion=0, # Molecular diffusion, in [m2/day]
)

mbt_object = mbt.Mibitrans(hydro, att, source, model)
mbt_object.hydrological_parameters.velocity = 0.1
mbt_results_v01 = mbt_object.run()

ana_object = mbt.Anatrans(hydro, att, source, model)
ana_object.hydrological_parameters.velocity = 0.1
ana_results_v01 = ana_object.run()

mbt_object.hydrological_parameters.velocity = 0.2
mbt_results_v02 = mbt_object.run()

ana_object.hydrological_parameters.velocity = 0.2
ana_results_v02 = ana_object.run()

In [ ]:
mbt_results_v01.centerline(time=3*365, color=colors_mbt[0], lw=3, alpha=0.7, label="mbt v=0.1, t=3y")
mbt_results_v02.centerline(time=3*365, color=colors_mbt[1], lw=3, alpha=0.7, label="mbt v=0.2, t=3y")
mbt_results_v01.centerline(time=6*365, color=colors_mbt[2], lw=2, linestyle="--", label="mbt v=0.1, t=6y")
mbt_results_v02.centerline(time=6*365, color=colors_mbt[3], lw=3, linestyle="--", label="mbt v=0.2, t=6y")
ana_results_v01.centerline(time=3*365, color=colors_ana[0], lw=3, alpha=0.7, label="ana v=0.1, t=3y")
ana_results_v02.centerline(time=3*365, color=colors_ana[1], lw=3, alpha=0.7, label="ana v=0.2, t=3y")
ana_results_v01.centerline(time=6*365, color=colors_ana[3], lw=2, linestyle="--", label="ana v=0.1, t=6y")
ana_results_v02.centerline(time=6*365, color=colors_ana[4], lw=3, linestyle="--", label="ana v=0.2, t=6y")

plt.xlim(-10, 600)
plt.title("Differences between Mibitrans and Anatrans for various flow velocities")
plt.legend()
plt.show()

The curves for $v=0.1m/s$ at $t=6y$ and $v=0.2m/s$ at $t=3y$ of Mibitrans are expected to overlap. Diffusion $=0$ and thus with twice as low velocity, it takes twice as long for the advective front to get into the same position. The distributions for Anatrans also show this overlap, implying that regardless of flow velocity, the difference between Anatrans and Mibitrans remain the same.

However, changing the transverse dispersivity does change the differences between the models, as is shown below.

In [ ]:
# Varying transverse horizontal dispersivities
alpha_y_list = [2, 1, 0.2, 0.1]
# And corresponding transverse vertical dispersivities
alpha_z_list = [0.05, 0.01, 0.005, 0.001]
output_mbt = []
output_ana = []

mbt_object_alpha = mbt.Mibitrans(hydro, att, source, model)
ana_object_alpha = mbt.Anatrans(hydro, att, source, model)

for i in range(len(alpha_y_list)):
    hydro = mbt.HydrologicalParameters(
        velocity=0.277,             # [m/d]
        porosity=0.25,              # [-]
        alpha_x=10,                 # [m]
        alpha_y=alpha_y_list[i],    # [m]
        alpha_z=alpha_z_list[i],    # [m]
        diffusion=0,                # [m2/day]
    )

    mbt_object_alpha.hydrological_parameters = hydro
    mbt_results_alpha = mbt_object_alpha.run()

    ana_object_alpha.hydrological_parameters = hydro
    ana_results_alpha = ana_object_alpha.run()

    output_mbt.append(mbt_results_alpha)
    output_ana.append(ana_results_alpha)

In [ ]:
for i in range(len(alpha_y_list)):
    output_mbt[i].centerline(
        time=5*365,
        color=colors_mbt[i],
        lw=2.5,
        label=fr"mbt, $\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )
    output_ana[i].centerline(
        time=5*365,
        color=colors_ana[i],
        linestyle="--",
        lw=2,
        label=fr"ana, $\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )

plt.legend(bbox_to_anchor=(1,1))
plt.xlim((-10, 700))

In [ ]:
x_pos = 150

for i in range(len(alpha_y_list)):
    output_mbt[i].transverse(
        x_position=x_pos,
        color=colors_mbt[i],
        lw=2.5,
        label=fr"mbt, $\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )
    output_ana[i].transverse(
        x_position=x_pos,
        color=colors_ana[i],
        linestyle=":",
        lw=2,
        label=fr"ana, $\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )

plt.legend(bbox_to_anchor=(1,1))
plt.xlim(-60,60)
plt.ylim(-0.2,11)
plt.show()

In [ ]:
x_pos = 120

for i in range(len(alpha_y_list)):
    output_mbt[i].breakthrough(
        x_position=x_pos,
        color=colors_mbt[i],
        lw=2.5,
        label=fr"mbt, $\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )
    output_ana[i].breakthrough(
        x_position=x_pos,
        color=colors_ana[i],
        linestyle="--",
        lw=2,
        label=fr"ana, $\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )

plt.legend(bbox_to_anchor=(1,0.85))
plt.xlim(-10,2000)
plt.ylim(-0.2,11)
plt.show()

Differences are better visualized as the relative and/or absolute difference between the two models.

In [ ]:
colors = ["maroon", "red", "orangered", "coral"]
time_step = len(mbt_object.t)//2 - 1
y_position = len(mbt_object.y)//2
print("plotting at t =", mbt_object.t[time_step] / 365, "years and a y-position =", mbt_object.y[y_position], "m")

for i in range(len(alpha_y_list)):
    diff = (output_ana[i].cxyt - output_mbt[i].cxyt) / output_mbt[i].cxyt
    plt.plot(
        mbt_object.x,
        diff[time_step,y_position,:],
        color=colors[i],
        lw=2,
        label=fr"$\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )

plt.xlabel("Distance from source [m]")
plt.ylabel("Relative difference")
plt.title("Relative difference between Mibitrans and Anatrans models")
plt.legend()
plt.show()

for i in range(len(alpha_y_list)):
    diff = output_ana[i].cxyt - output_mbt[i].cxyt
    plt.plot(
        mbt_object.x,
        diff[time_step,y_position,:],
        color=colors[i],
        lw=2,
        label=fr"$\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$"
    )

plt.xlabel("Distance from source [m]")
plt.ylabel("Absolute difference")
plt.title("Absolute difference between Mibitrans and Anatrans models")
plt.legend()
plt.show()

In [ ]:
time_step = len(mbt_object.t)//2 - 1

fig, ax = plt.subplots(2,2, sharex=True, sharey=True)
plot_list = []
i=0

for j in range(len(ax[:,0])):
    for k in range(len(ax[0,:])):
        diff = output_ana[i].relative_cxyt - output_mbt[i].relative_cxyt
        plot_list.append(ax[j,k].pcolormesh(
            mbt_object.x[:len(mbt_object.x)//4],
            mbt_object.y[len(mbt_object.y)//5:-len(mbt_object.y)//5],
            diff[time_step,len(mbt_object.y)//5:-len(mbt_object.y)//5,:len(mbt_object.x)//4],
            vmin=-0.3,
            vmax=0.3,
            cmap="seismic",
            shading="gouraud"
        ))
        ax[j,k].set_title(fr"$\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$", fontsize=9)
        i+=1

fig.colorbar(plot_list[0], ax=ax, orientation="vertical", label="Absolute difference in relative concentration")
fig.supxlabel("Distance from source [m]", fontsize=10)
fig.supylabel("y-position [m]", fontsize=10)
fig.suptitle("Difference between Mibitrans and Anatrans model for various dispersivities", fontsize=11)
plt.show()

In [ ]:
time_step = len(mbt_object.t)//2 - 1
fig, ax = plt.subplots(2,2, sharex=True, sharey=True)
plot_list = []
i=0

for j in range(len(ax[:,0])):
    for k in range(len(ax[0,:])):
        diff = output_ana[i].relative_cxyt - output_mbt[i].relative_cxyt
        plot_list.append(ax[j,k].pcolormesh(
            mbt_object.x,
            mbt_object.y,
            diff[time_step,:,:],
            vmin=-0.3,
            vmax=0.3,
            cmap="seismic",
            shading="gouraud"
        ))
        ax[j,k].set_title(fr"$\alpha_y={alpha_y_list[i]}m, \alpha_z={alpha_z_list[i]}m$", fontsize=9)
        # To show differences more clearly, x-limit is 1/4 of model extent
        ax[j,k].set_xlim(-1, mbt_object.x[len(mbt_object.x)//4])
        # Also limit y-extent
        ax[j,k].set_ylim(mbt_object.y[len(mbt_object.y)//5], mbt_object.y[-len(mbt_object.y)//5])
        i+=1

fig.colorbar(plot_list[0], ax=ax, orientation="vertical", label="Absolute difference in relative concentration")
fig.supxlabel("Distance from source [m]", fontsize=10)
fig.supylabel("y-position [m]", fontsize=10)
fig.suptitle("Difference between Mibitrans and Anatrans model for various dispersivities", fontsize=11)
plt.show()

Domenico, P. A., *An analytical model for multidimensional transport of a decaying contaminant species*, Journal of Hydrology, 91 (1), 49–58, doi:10.1016/0022-1694(87)90127-2, 1987.

Guyonnet, D., & Neville, C., *Dimensionless analysis of two analytical solutions for 3-D solute transport in groundwater*. Journal of Contaminant Hydrology, 75(1), 141–153, https://doi.org/10.1016/j.jconhyd.2004.06.004, 2004.
